In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().parent
SPLIT_DIR = PROJECT_ROOT / "fine_tuning" / "data_splits"
ARTIFACT_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")

development_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

development_df["fold"] = -1

for fold, (_, val_idx) in enumerate(
    skf.split(development_df, development_df["TARGET"])
):
    development_df.loc[val_idx, "fold"] = fold

fold_assignments = development_df[["SK_ID_CURR", "TARGET", "fold"]]

fold_assignments.to_csv(
    ARTIFACT_DIR / "cv_fold_assignments.csv",
    index=False
)

print("Development rows:", len(development_df))
print("Fold distribution:")
print(development_df["fold"].value_counts().sort_index())

print("\nDefault rate by fold:")
print(development_df.groupby("fold")["TARGET"].mean().round(4))

C:\Users\nitingowda\AppData\Local\Temp\ipykernel_11788\2884160597.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  development_df["fold"] = -1


Development rows: 261384
Fold distribution:
fold
0    52277
1    52277
2    52277
3    52277
4    52276
Name: count, dtype: int64

Default rate by fold:
fold
0    0.0807
1    0.0807
2    0.0807
3    0.0807
4    0.0807
Name: TARGET, dtype: float64


In [4]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from feature_engineering import create_engineered_features

# Load selected 108 features
PROJECT_ROOT = Path.cwd().parent
ARTIFACT_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"

with open(ARTIFACT_DIR / "feature_selection_config.json") as f:
    feature_config = json.load(f)

selected_features = feature_config["reduced_features"]

print("Selected features:", len(selected_features))

# Five-fold cross-validation
fold_results = []

for fold in range(5):
    print(f"\nRunning fold {fold + 1}/5...")

    outer_train = development_df[development_df["fold"] != fold]
    outer_val = development_df[development_df["fold"] == fold]

    X_outer = create_engineered_features(
        outer_train.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
    )[selected_features]
    y_outer = outer_train["TARGET"]

    X_val_fold = create_engineered_features(
        outer_val.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
    )[selected_features]
    y_val_fold = outer_val["TARGET"]

    # Inner split for early stopping
    X_inner_train, X_inner_val, y_inner_train, y_inner_val = train_test_split(
        X_outer, y_outer,
        test_size=0.10,
        stratify=y_outer,
        random_state=42
    )

    numeric_features = X_inner_train.select_dtypes(include=np.number).columns
    categorical_features = X_inner_train.select_dtypes(exclude=np.number).columns

    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", dtype=np.float32))
        ]), categorical_features)
    ], sparse_threshold=1.0)

    X_inner_train_t = preprocessor.fit_transform(X_inner_train)
    X_inner_val_t = preprocessor.transform(X_inner_val)
    X_outer_val_t = preprocessor.transform(X_val_fold)

    model = XGBClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=3.375,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_inner_train_t,
        y_inner_train,
        eval_set=[(X_inner_val_t, y_inner_val)],
        verbose=False
    )

    probabilities = model.predict_proba(X_outer_val_t)[:, 1]

    fold_results.append({
        "fold": fold + 1,
        "best_iteration": model.best_iteration,
        "roc_auc": roc_auc_score(y_val_fold, probabilities),
        "average_precision": average_precision_score(y_val_fold, probabilities)
    })

    print(
        f"ROC-AUC: {fold_results[-1]['roc_auc']:.4f} | "
        f"AP: {fold_results[-1]['average_precision']:.4f}"
    )

# Final results
cv_results = pd.DataFrame(fold_results)

print("\nCross-validation results:")
display(cv_results)

print("\nMean CV performance:")
display(cv_results[["roc_auc", "average_precision"]].agg(["mean", "std"]))

Selected features: 108

Running fold 1/5...
ROC-AUC: 0.7644 | AP: 0.2400

Running fold 2/5...
ROC-AUC: 0.7620 | AP: 0.2500

Running fold 3/5...
ROC-AUC: 0.7637 | AP: 0.2510

Running fold 4/5...
ROC-AUC: 0.7673 | AP: 0.2557

Running fold 5/5...
ROC-AUC: 0.7593 | AP: 0.2493

Cross-validation results:


,fold,best_iteration,roc_auc,average_precision
0,1,191,0.764413,0.240022
1,2,471,0.762028,0.249951
2,3,275,0.763740,0.250973
3,4,271,0.767308,0.255685
4,5,189,0.759271,0.249349



Mean CV performance:


,roc_auc,average_precision
mean,0.763352,0.249196
std,0.002972,0.005701


In [5]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from feature_engineering import create_engineered_features

# Load both feature sets
PROJECT_ROOT = Path.cwd().parent
ARTIFACT_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"

with open(ARTIFACT_DIR / "feature_selection_config.json") as f:
    config = json.load(f)

feature_sets = {
    "full_129": config["full_features"],
    "reduced_108": config["reduced_features"]
}

comparison_results = []

for feature_name, features in feature_sets.items():
    print(f"\nEvaluating {feature_name} ({len(features)} features)")

    for fold in range(5):
        outer_train = development_df[development_df["fold"] != fold]
        outer_val = development_df[development_df["fold"] == fold]

        X_outer = create_engineered_features(
            outer_train.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
        )[features]
        y_outer = outer_train["TARGET"]

        X_val = create_engineered_features(
            outer_val.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
        )[features]
        y_val = outer_val["TARGET"]

        X_train, X_inner_val, y_train, y_inner_val = train_test_split(
            X_outer, y_outer,
            test_size=0.10,
            stratify=y_outer,
            random_state=42
        )

        numeric = X_train.select_dtypes(include=np.number).columns
        categorical = X_train.select_dtypes(exclude=np.number).columns

        preprocessor = ColumnTransformer([
            ("num", SimpleImputer(strategy="median", add_indicator=True), numeric),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore", dtype=np.float32
                ))
            ]), categorical)
        ], sparse_threshold=1.0)

        X_train_t = preprocessor.fit_transform(X_train)
        X_inner_val_t = preprocessor.transform(X_inner_val)
        X_val_t = preprocessor.transform(X_val)

        model = XGBClassifier(
            n_estimators=1500,
            learning_rate=0.05,
            max_depth=5,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            scale_pos_weight=3.375,
            early_stopping_rounds=50,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train_t, y_train,
            eval_set=[(X_inner_val_t, y_inner_val)],
            verbose=False
        )

        probabilities = model.predict_proba(X_val_t)[:, 1]

        comparison_results.append({
            "feature_set": feature_name,
            "fold": fold + 1,
            "best_iteration": model.best_iteration,
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities)
        })

        print(
            f"Fold {fold + 1}/5 | "
            f"ROC-AUC: {comparison_results[-1]['roc_auc']:.4f} | "
            f"AP: {comparison_results[-1]['average_precision']:.4f}"
        )

comparison_df = pd.DataFrame(comparison_results)

print("\nMean and standard deviation:")
display(
    comparison_df.groupby("feature_set")[
        ["roc_auc", "average_precision", "best_iteration"]
    ].agg(["mean", "std"]).round(4)
)

comparison_df.to_csv(
    ARTIFACT_DIR / "feature_set_cv_comparison.csv",
    index=False
)


Evaluating full_129 (129 features)
Fold 1/5 | ROC-AUC: 0.7643 | AP: 0.2409
Fold 2/5 | ROC-AUC: 0.7625 | AP: 0.2489
Fold 3/5 | ROC-AUC: 0.7643 | AP: 0.2526
Fold 4/5 | ROC-AUC: 0.7675 | AP: 0.2570
Fold 5/5 | ROC-AUC: 0.7637 | AP: 0.2535

Evaluating reduced_108 (108 features)
Fold 1/5 | ROC-AUC: 0.7644 | AP: 0.2400
Fold 2/5 | ROC-AUC: 0.7620 | AP: 0.2500
Fold 3/5 | ROC-AUC: 0.7637 | AP: 0.2510
Fold 4/5 | ROC-AUC: 0.7673 | AP: 0.2557
Fold 5/5 | ROC-AUC: 0.7593 | AP: 0.2493

Mean and standard deviation:


roc_auc         average_precision         best_iteration          
               mean     std              mean     std           mean       std
feature_set                                                                   
full_129     0.7645  0.0018            0.2506  0.0061          360.2  125.0608
reduced_108  0.7634  0.0030            0.2492  0.0057          279.4  114.8773

In [6]:
import numpy as np
import pandas as pd
import json

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
from xgboost import XGBClassifier

# Load full feature set and untouched test data
with open(ARTIFACT_DIR / "feature_selection_config.json") as f:
    config = json.load(f)

full_features = config["full_features"]

test_df = pd.read_csv(SPLIT_DIR / "test.csv")

X_dev = create_engineered_features(
    development_df.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
)[full_features]
y_dev = development_df["TARGET"]

X_test = create_engineered_features(
    test_df.drop(columns=["TARGET", "SK_ID_CURR"])
)[full_features]
y_test = test_df["TARGET"]

# Use median best iteration from full-feature CV
full_cv = comparison_df[comparison_df["feature_set"] == "full_129"]
final_estimators = int(full_cv["best_iteration"].median()) + 1

print("Final estimators:", final_estimators)

# Fit preprocessing on all development data
numeric = X_dev.select_dtypes(include=np.number).columns
categorical = X_dev.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", dtype=np.float32))
    ]), categorical)
], sparse_threshold=1.0)

X_dev_t = preprocessor.fit_transform(X_dev)
X_test_t = preprocessor.transform(X_test)

# Train final model
final_model = XGBClassifier(
    n_estimators=final_estimators,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    scale_pos_weight=3.375,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_dev_t, y_dev, verbose=False)

# One-time test evaluation
test_probabilities = final_model.predict_proba(X_test_t)[:, 1]

print("\nFINAL TEST RESULTS")
print("ROC-AUC:", round(roc_auc_score(y_test, test_probabilities), 4))
print("Average Precision:", round(average_precision_score(y_test, test_probabilities), 4))

Final estimators: 351

FINAL TEST RESULTS
ROC-AUC: 0.7703
Average Precision: 0.2565


In [7]:
import joblib
import json
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, MODEL_DIR / "v2_xgboost_model.joblib")
joblib.dump(preprocessor, MODEL_DIR / "v2_preprocessor.joblib")

with open(MODEL_DIR / "v2_model_metrics.json", "w") as f:
    json.dump({
        "model": "XGBoost",
        "features": len(full_features),
        "n_estimators": final_estimators,
        "roc_auc": 0.7703,
        "average_precision": 0.2565,
        "test_rows": len(test_df),
        "training_rows": len(development_df)
    }, f, indent=4)

print("V2 model, preprocessor and metrics saved.")

V2 model, preprocessor and metrics saved.


In [8]:
import numpy as np
import pandas as pd
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    confusion_matrix
)
from xgboost import XGBClassifier
from feature_engineering import create_engineered_features

# Load development data and fold assignments
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
development_df = pd.concat([train_df, val_df], ignore_index=True)

fold_df = pd.read_csv(ARTIFACT_DIR / "cv_fold_assignments.csv")
fold_map = fold_df.set_index("SK_ID_CURR")["fold"]
development_df["fold"] = development_df["SK_ID_CURR"].map(fold_map)

with open(ARTIFACT_DIR / "feature_selection_config.json") as f:
    config = json.load(f)

full_features = config["full_features"]

oof_probabilities = np.zeros(len(development_df))

# Generate out-of-fold predictions
for fold in range(5):
    print(f"Running fold {fold + 1}/5...")

    outer_train = development_df[development_df["fold"] != fold]
    outer_val = development_df[development_df["fold"] == fold]

    X_outer = create_engineered_features(
        outer_train.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
    )[full_features]
    y_outer = outer_train["TARGET"]

    X_val = create_engineered_features(
        outer_val.drop(columns=["TARGET", "SK_ID_CURR", "fold"])
    )[full_features]
    y_val = outer_val["TARGET"]

    X_inner_train, X_inner_val, y_inner_train, y_inner_val = train_test_split(
        X_outer, y_outer, test_size=0.10,
        stratify=y_outer, random_state=42
    )

    numeric = X_inner_train.select_dtypes(include=np.number).columns
    categorical = X_inner_train.select_dtypes(exclude=np.number).columns

    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="median", add_indicator=True), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", dtype=np.float32))
        ]), categorical)
    ], sparse_threshold=1.0)

    X_train_t = preprocessor.fit_transform(X_inner_train)
    X_inner_val_t = preprocessor.transform(X_inner_val)
    X_val_t = preprocessor.transform(X_val)

    model = XGBClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=3.375,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_t, y_inner_train,
        eval_set=[(X_inner_val_t, y_inner_val)],
        verbose=False
    )

    oof_probabilities[outer_val.index] = model.predict_proba(X_val_t)[:, 1]

    print(f"Best iteration: {model.best_iteration}")

# Save OOF predictions
oof_df = pd.DataFrame({
    "SK_ID_CURR": development_df["SK_ID_CURR"],
    "TARGET": development_df["TARGET"],
    "probability": oof_probabilities
})

oof_df.to_csv(ARTIFACT_DIR / "v2_oof_predictions.csv", index=False)

# Compare candidate thresholds
threshold_results = []

for threshold in np.arange(0.05, 0.501, 0.01):
    predictions = (oof_probabilities >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(development_df["TARGET"], predictions, zero_division=0),
        "recall": recall_score(development_df["TARGET"], predictions),
        "f1": f1_score(development_df["TARGET"], predictions),
        "f2": fbeta_score(development_df["TARGET"], predictions, beta=2)
    })

threshold_df = pd.DataFrame(threshold_results)

print("\nTop 10 thresholds by F2:")
display(threshold_df.sort_values("f2", ascending=False).head(10).round(4))

best_threshold = float(
    threshold_df.loc[threshold_df["f2"].idxmax(), "threshold"]
)

print("\nProvisional F2-optimal threshold:", best_threshold)

C:\Users\nitingowda\AppData\Local\Temp\ipykernel_11788\4281855495.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  development_df["fold"] = development_df["SK_ID_CURR"].map(fold_map)


Running fold 1/5...
Best iteration: 190
Running fold 2/5...
Best iteration: 340
Running fold 3/5...
Best iteration: 350
Running fold 4/5...
Best iteration: 380
Running fold 5/5...
Best iteration: 541

Top 10 thresholds by F2:


,threshold,precision,recall,f1,f2
17,0.22,0.1682,0.6956,0.2709,0.4275
16,0.21,0.1634,0.7160,0.2661,0.4272
18,0.23,0.1729,0.6739,0.2752,0.4266
19,0.24,0.1780,0.6540,0.2798,0.4261
15,0.20,0.1586,0.7361,0.2609,0.4259
20,0.25,0.1829,0.6335,0.2838,0.4243
14,0.19,0.1535,0.7548,0.2551,0.4232
21,0.26,0.1876,0.6146,0.2875,0.4224
22,0.27,0.1931,0.5979,0.2919,0.4212
13,0.18,0.1487,0.7744,0.2494,0.4205



Provisional F2-optimal threshold: 0.22
